# Price curves 03: censored fits by year

Issue #86 fits the parametric families by censored maximum likelihood, using
censored slots as bounds instead of dropping them. 2021 has about a third of its
slots censored, so its fit leans most on the noise model (ADR-0003).

This notebook fits each training year separately and shows two sets of numbers
side by side:

- **censored:** maximum likelihood on every slot, censored ones as bounds
- **exact only:** least squares on the exact slots alone, which is what the
  fit did before #86

Both the raw and the fuel-adjusted target are shown.

In [ ]:
import os
from pathlib import Path

from fleet_sim.prices.config import load_config
from fleet_sim.prices.curves import FAMILIES
from fleet_sim.prices.dataset import build_dataset, selection_data
from fleet_sim.prices.targets import FuelAdjustedTarget, RawTarget

# The config's cache_dir is relative to the repo root, so run from there.
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)

config = load_config(Path("calibration/price_curves.toml"))
selection = selection_data(build_dataset(config), config.splits)
PARAMETRIC = ["cubic", "signed_quadratic", "cubic_offset", "exponential"]
TARGETS = {"raw": RawTarget(per_year=True), "fuel_adjusted": FuelAdjustedTarget(per_year=True)}

In [ ]:
def fit_year(family, target, observations):
    r = [o.residual_load_gw for o in observations]
    bounds = [target.to_target_bounds(o) for o in observations]
    return family.fit(r, [b[0] for b in bounds], [b[1] for b in bounds])


def row(label, values):
    return f"    {label:<11}" + "".join(f"{v:>12.4g}" for v in values)


def show(target_name):
    target = TARGETS[target_name]
    for family_name in PARAMETRIC:
        family = FAMILIES[family_name]
        print(f"{target_name} / {family_name}")
        for year in selection.years:
            rows = [o for o in selection.observations if o.year == year]
            share = sum(o.censored for o in rows) / len(rows)
            censored = fit_year(family, target, rows).model_dump()
            exact = fit_year(family, target, [o for o in rows if not o.censored]).model_dump()
            print(f"  {year}, {share:.0%} censored")
            print(f"    {'':<11}" + "".join(f"{n:>12}" for n in censored))
            print(row("censored", censored.values()))
            print(row("exact only", exact.values()))
        print()

## Raw target, π (£/MWh)

In [ ]:
show("raw")

## Fuel-adjusted target, π / k

In [ ]:
show("fuel_adjusted")

## What moves

**2021 and 2022, about a third censored.** The censored fits are steeper and have
a larger σ. Counting the censored slots as "at least this much" pulls the top of
the curve up:

| Target, family | 2021: censored vs exact only | 2022: censored vs exact only |
| --- | --- | --- |
| raw, cubic a | 0.0129 vs 0.0098 (+32%) | 0.0168 vs 0.0135 (+24%) |
| fuel-adjusted, cubic a | +12% | +16% |
| raw, cubic σ | 84 vs 65 £/MWh | 129 vs 104 £/MWh |

The raw exponential fitted to the exact 2021 slots alone is degenerate
(α = −790, β = 470, γ = −0.53). With the censored slots it becomes a convex
curve (γ = 0.15).

**2023, 5% censored.** Parameters move a few per cent.

**2024, 5 censored slots.** The one-parameter fits barely move. The offset cubic
and the exponential move a little: the 5 slots are extreme prices, so even five
of them shift a three-parameter fit.

**What this rests on.** 2021 leans on the noise model the most. ADR-0003 notes
that σ is too small at high R in 2021–2022, where most censored slots are. So
these fits still place the censored prices too low, and the true rise is
probably steeper still.